# image

In [ ]:
from pathlib import Path
import json
import os
from dotenv import load_dotenv
import fitz

# -----------------------------
# Locate Repository Root
# -----------------------------
repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

pdf_path = os.getenv("PDF_PATH")

if not pdf_path:
    raise ValueError("PDF_PATH not found inside .env")

pdf_path = Path(pdf_path)

if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not pdf_path.exists():
    raise FileNotFoundError(pdf_path)

doc = fitz.open(pdf_path)

print(f"Total Pages : {len(doc)}")

# pages_dir = repo_root / "pages"
# print(f"Pages Directory : {pages_dir}")

Total Pages : 1170
Pages Directory : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\pages


In [11]:
from pathlib import Path
import fitz
import json
import os
import re
from dotenv import load_dotenv

# ==================================================
# Find Project Root
# ==================================================

repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

print("\n========== PROJECT ==========")
print("Repo Root :", repo_root)

# ==================================================
# Load .env
# ==================================================

env_file = repo_root / ".env"

if env_file.exists():
    load_dotenv(env_file)
    print(".env Loaded :", env_file)
else:
    print("WARNING: .env not found")

# ==================================================
# Read Environment Variables
# ==================================================

pdf_path_env = os.getenv(
    "PDF_PATH",
    "Backend/Books/sample_book.pdf"
)

schema_path_env = os.getenv(
    "SCHEMA_PATH",
    "Backend/Output/book_structure.json"
)

output_path_env = os.getenv(
    "OUTPUT_PATH",
    "Backend/Output"
)

print("\n========== CONFIG ==========")
print("PDF_PATH    =", pdf_path_env)
print("SCHEMA_PATH =", schema_path_env)
print("OUTPUT_PATH =", output_path_env)

# ==================================================
# Build Absolute Paths
# ==================================================

pdf_path = Path(pdf_path_env)
schema_path = Path(schema_path_env)
output_root = Path(output_path_env)

if not pdf_path.is_absolute():
    pdf_path = repo_root / pdf_path

if not schema_path.is_absolute():
    schema_path = repo_root / schema_path

if not output_root.is_absolute():
    output_root = repo_root / output_root

print("\n========== ABSOLUTE PATHS ==========")
print("PDF File    :", pdf_path)
print("Schema File :", schema_path)
print("Output Dir  :", output_root)

# ==================================================
# Validate PDF
# ==================================================

if not pdf_path.exists():
    raise FileNotFoundError(
        f"\nPDF NOT FOUND:\n{pdf_path}"
    )

# ==================================================
# Open PDF
# ==================================================

doc = fitz.open(pdf_path)

print("\n========== PDF ==========")
print("PDF Loaded Successfully")
print("Total Pages :", len(doc))

# ==================================================
# Validate Schema
# ==================================================

if not schema_path.exists():

    print("\nWARNING:")
    print("Schema file not found.")
    print("Expected:", schema_path)

    print("\nCreate your schema first.")
    print("Stopping extraction.")

    raise SystemExit

# ==================================================
# Load Schema
# ==================================================

with open(
    schema_path,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)

print("\nSchema Loaded Successfully")

# ==================================================
# Support Multiple Schema Formats
# ==================================================

if isinstance(schema, dict):
    chapters = schema.get("chapters", [])
elif isinstance(schema, list):
    chapters = schema
else:
    raise ValueError(
        "Unsupported schema format"
    )

print(
    f"Chapters Found : {len(chapters)}"
)

# ==================================================
# Create Output Folder
# ==================================================

output_root.mkdir(
    parents=True,
    exist_ok=True
)

# ==================================================
# Caption Detection Patterns
# ==================================================

caption_patterns = [
    r"Figure\s+\d+[\.\d]*.*",
    r"FIGURE\s+\d+[\.\d]*.*",
    r"Fig\.\s*\d+[\.\d]*.*"
]

# ==================================================
# Statistics
# ==================================================

total_images = 0

# ==================================================
# Process Chapters
# ==================================================

for chapter in chapters:

    chapter_number = chapter.get(
        "chapter_number",
        chapter.get("chapter", "Unknown")
    )

    chapter_title = chapter.get(
        "chapter_title",
        chapter.get("title", "Unknown")
    )

    print(
        f"\nProcessing Chapter "
        f"{chapter_number}"
    )

    print(
        f"Title: {chapter_title}"
    )

    safe_chapter = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        str(chapter_title)
    )

    chapter_folder = (
        output_root /
        f"Chapter_{chapter_number}_{safe_chapter}"
    )

    chapter_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    topics = chapter.get(
        "topics",
        []
    )

    for topic in topics:

        topic_title = topic.get(
            "topic_title",
            topic.get(
                "title",
                "Unknown Topic"
            )
        )

        start_page = topic.get(
            "start_page"
        )

        end_page = topic.get(
            "end_page"
        )

        if (
            start_page is None
            or end_page is None
        ):
            print(
                f"Skipping topic "
                f"{topic_title}"
            )
            continue

        print(
            f"   Topic : {topic_title}"
        )

        print(
            f"   Pages : "
            f"{start_page} -> {end_page}"
        )

        safe_topic = re.sub(
            r'[<>:"/\\|?*]',
            "_",
            str(topic_title)
        )

        topic_folder = (
            chapter_folder /
            safe_topic
        )

        images_folder = (
            topic_folder /
            "images"
        )

        images_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        figures_metadata = []

        # ==========================================
        # Process Pages
        # ==========================================

        for page_num in range(
            start_page,
            end_page + 1
        ):

            if page_num >= len(doc):
                continue

            page = doc[page_num]

            page_text = page.get_text()

            captions_found = []

            for pattern in caption_patterns:

                matches = re.findall(
                    pattern,
                    page_text,
                    flags=re.IGNORECASE
                )

                captions_found.extend(
                    matches
                )

            images = page.get_images(
                full=True
            )

            if images:

                print(
                    f"      Page "
                    f"{page_num+1}: "
                    f"{len(images)} image(s)"
                )

            for idx, img in enumerate(
                images
            ):

                try:

                    xref = img[0]

                    pix = fitz.Pixmap(
                        doc,
                        xref
                    )

                    if pix.n >= 5:
                        pix = fitz.Pixmap(
                            fitz.csRGB,
                            pix
                        )

                    image_name = (
                        f"page_"
                        f"{page_num+1}"
                        f"_img_"
                        f"{idx+1}.png"
                    )

                    image_path = (
                        images_folder /
                        image_name
                    )

                    pix.save(
                        str(image_path)
                    )

                    caption = ""

                    if captions_found:
                        caption = (
                            captions_found[
                                min(
                                    idx,
                                    len(
                                        captions_found
                                    ) - 1
                                )
                            ]
                        )

                    figures_metadata.append(
                        {
                            "chapter":
                            chapter_number,

                            "chapter_title":
                            chapter_title,

                            "topic":
                            topic_title,

                            "page":
                            page_num + 1,

                            "image":
                            image_name,

                            "caption":
                            caption,

                            "path":
                            str(
                                image_path
                            )
                        }
                    )

                    total_images += 1

                    print(
                        "         Saved:",
                        image_name
                    )

                except Exception as e:

                    print(
                        "Image Error:",
                        e
                    )

        metadata_file = (
            topic_folder /
            "figures.json"
        )

        with open(
            metadata_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                figures_metadata,
                f,
                indent=4,
                ensure_ascii=False
            )

        print(
            "   Metadata Saved:",
            metadata_file
        )

# ==================================================
# Final Summary
# ==================================================

print("\n=================================")
print("EXTRACTION COMPLETE")
print("=================================")
print(
    "Total Images Extracted:",
    total_images
)
print(
    "Output Folder:",
    output_root
)


========== PROJECT ==========
Repo Root : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline
.env Loaded : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\.env

========== CONFIG ==========
PDF_PATH    = Backend/Books/sample_book.pdf
SCHEMA_PATH = Backend/Output/book_structure.json
OUTPUT_PATH = Backend/Output

========== ABSOLUTE PATHS ==========
PDF File    : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Books\sample_book.pdf
Schema File : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Output\book_structure.json
Output Dir  : c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Output

========== PDF ==========
PDF Loaded Successfully
Total Pages : 1170

Schema file not found.
Expected: c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\Backend\Output\book_structure.json

Create your schema first.
Stopping extraction.


SystemExit: 

c:\Users\hp\Desktop\AI-PDF-Chapter-Topic-Image-Extraction-Pipeline\venv\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
